# Assignment 8: Normalising Flows

This notebook covers **Normalising Flows** — a class of generative models based on exact likelihood computation through invertible transformations.

Topics covered (Lecture 8):
- The change of variables formula for exact density evaluation
- Invertible transformations and their Jacobians
- Affine coupling layers as tractable invertible blocks
- RealNVP: stacking coupling layers with alternating masks
- Training normalising flows by maximising likelihood

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

np.random.seed(42)
plt.rcParams['figure.figsize'] = (9, 6)

---
## Part 1: Change of Variables Formula

If $x = f(z)$ is a differentiable invertible mapping, then the change of variables formula gives us the density of $x$:

$$\log p_X(x) = \log p_Z(f^{-1}(x)) + \log |\det J_{f^{-1}}(x)|$$
$$\phantom{\log p_X(x)} = \log p_Z(z) - \log |\det J_f(z)|$$

where $J_f(z)$ is the Jacobian of $f$ at $z$.

Key requirements:
1. $f$ must be **invertible** (bijective)
2. $\log |\det J|$ must be **tractable** to compute

The intuition: if $f$ expands volume (large $|\det J_f|$), probability mass is spread out and density decreases. If $f$ contracts volume, density increases.

In [ ]:
def change_of_variables_log_prob(log_pz, log_det_jac_f):
    """Compute log p_X(x) using the change of variables formula.

    log p_X(x) = log p_Z(z) - log |det J_f(z)|

    Args:
        log_pz          : float or np.ndarray, log p_Z(z)
        log_det_jac_f   : float or np.ndarray, log |det J_f(z)| (of f: z -> x)

    Returns:
        float or np.ndarray, log p_X(x)
    """
    # TODO: return log_pz - log_det_jac_f
    return log_pz - log_det_jac_f

In [ ]:
# Sanity check: 1D affine transform x = 2*z + 3 with p_Z = N(0, 1)
# J_f = 2, so log|det J_f| = log(2)
# p_X should be N(3, 4) — mean=3, variance=4 (std=2)

z_demo = np.linspace(-4, 4, 300)
x_demo = 2 * z_demo + 3  # f(z) = 2z + 3

log_pz_demo = norm.logpdf(z_demo, 0, 1)          # log N(z | 0, 1)
log_det_jac_demo = np.log(2.0)                    # log |det J_f| = log(2)

log_px_flow   = change_of_variables_log_prob(log_pz_demo, log_det_jac_demo)
log_px_analytic = norm.logpdf(x_demo, 3, 2)      # log N(x | 3, 2)

max_err = np.max(np.abs(log_px_flow - log_px_analytic))
print(f'Max abs error vs analytic N(3,4): {max_err:.2e}  (should be ~0)')

plt.figure()
plt.plot(x_demo, np.exp(log_px_flow),    label='Change of variables', lw=2)
plt.plot(x_demo, np.exp(log_px_analytic), '--', label='Analytic N(3, 4)', lw=2)
plt.xlabel('x')
plt.ylabel('p_X(x)')
plt.title('Change of variables: f(z) = 2z + 3, p_Z = N(0,1)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

---
## Part 2: 1D Normalising Flow (Affine Transform)

The simplest normalising flow is a **1D affine transformation**: $x = \text{scale} \cdot z + \text{shift}$.

- **Forward** ($z \to x$): $x = e^{s} \cdot z + b$; $\log |\det J_f| = s$
- **Inverse** ($x \to z$): $z = (x - b) / e^{s}$; $\log |\det J_{f^{-1}}| = -s$

We parametrise the scale as $e^{s}$ (using `log_scale = s`) rather than using the scale directly. This ensures the scale is always positive without constraints, which is important for gradient-based optimisation.

In [ ]:
def affine_forward_1d(z, log_scale, shift):
    """Forward pass of 1D affine flow: x = exp(log_scale) * z + shift.

    Using log_scale (rather than scale) ensures positivity without constraints.

    Args:
        z         : np.ndarray (N,)
        log_scale : float
        shift     : float

    Returns:
        x            : np.ndarray (N,)
        log_det_jac  : float, log |det J_f| = log_scale (scalar, same for all z)
    """
    # TODO:
    # scale = np.exp(log_scale)
    # x = scale * z + shift
    # log_det_jac = log_scale  (log|scale| for all N points is just log_scale)
    # return x, log_det_jac
    scale = np.exp(log_scale)
    x = scale * z + shift
    log_det_jac = log_scale
    return x, log_det_jac


def affine_inverse_1d(x, log_scale, shift):
    """Inverse of 1D affine flow: z = (x - shift) / exp(log_scale).

    Args:
        x         : np.ndarray (N,)
        log_scale : float
        shift     : float

    Returns:
        z           : np.ndarray (N,)
        log_det_jac : float, log |det J_{f^{-1}}| = -log_scale
    """
    # TODO:
    # z = (x - shift) / np.exp(log_scale)
    # log_det_jac = -log_scale
    # return z, log_det_jac
    z = (x - shift) / np.exp(log_scale)
    log_det_jac = -log_scale
    return z, log_det_jac

In [ ]:
# Sanity check 1: forward then inverse should recover z
np.random.seed(0)
z_test = np.random.randn(100)
x_test, ldj_fwd = affine_forward_1d(z_test, log_scale=0.5, shift=2.0)
z_rec,  ldj_inv = affine_inverse_1d(x_test, log_scale=0.5, shift=2.0)

print(f'Roundtrip max error: {np.max(np.abs(z_test - z_rec)):.2e}  (should be ~0)')
print(f'log_det_fwd = {ldj_fwd:.4f},  log_det_inv = {ldj_inv:.4f}  (should sum to 0)')
print(f'Sum: {ldj_fwd + ldj_inv:.4f}')

In [ ]:
def flow_log_prob_1d(x, log_scale, shift):
    """Compute log p_X(x) for a 1D affine flow with standard normal base.

    Args:
        x         : np.ndarray (N,)
        log_scale : float
        shift     : float

    Returns:
        np.ndarray (N,), log probabilities
    """
    # TODO:
    # 1. Invert: z, log_det_inv = affine_inverse_1d(x, log_scale, shift)
    # 2. log_pz = norm.logpdf(z, 0, 1)  (log N(z | 0, 1))
    # 3. return log_pz + log_det_inv
    #    (change of variables: log p_X(x) = log p_Z(z) + log|det J_{f^{-1}}(x)|)
    z, log_det_inv = affine_inverse_1d(x, log_scale, shift)
    log_pz = norm.logpdf(z, 0, 1)
    return log_pz + log_det_inv

In [ ]:
# Sanity check 2: flow_log_prob_1d with log_scale=log(2), shift=3 should match N(3, 4)
x_grid = np.linspace(-5, 10, 400)

log_scale_true = np.log(2.0)  # scale=2 => N(3, 4)
shift_true     = 3.0

log_px_flow1d    = flow_log_prob_1d(x_grid, log_scale_true, shift_true)
log_px_analytic1d = norm.logpdf(x_grid, 3, 2)

print(f'Max error vs N(3,4): {np.max(np.abs(log_px_flow1d - log_px_analytic1d)):.2e}  (should be ~0)')

plt.figure()
plt.plot(x_grid, np.exp(log_px_flow1d),     label='1D affine flow', lw=2)
plt.plot(x_grid, np.exp(log_px_analytic1d), '--', label='Analytic N(3, 4)', lw=2)
plt.xlabel('x')
plt.ylabel('p_X(x)')
plt.title('1D affine flow log-prob vs analytic N(3, 4)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Grid search: fit log_scale and shift to N(3, 4) data
np.random.seed(7)
X_1d = np.random.randn(500) * 2 + 3  # samples from N(3, 4)

log_scale_vals = np.linspace(-1, 2, 50)
shift_vals     = np.linspace(0, 6, 50)

best_nll = np.inf
best_ls, best_sh = None, None

for ls in log_scale_vals:
    for sh in shift_vals:
        nll = -flow_log_prob_1d(X_1d, ls, sh).mean()
        if nll < best_nll:
            best_nll = nll
            best_ls, best_sh = ls, sh

print(f'Best log_scale = {best_ls:.3f}  (expected ≈ {np.log(2):.3f} = log(2))')
print(f'Best shift     = {best_sh:.3f}  (expected ≈ 3.0)')
print(f'Best NLL       = {best_nll:.4f}')

---
## Part 3: Composition of Flows

Multiple flows can be **composed**: $x = f_K \circ \cdots \circ f_1(z)$

The log-det-Jacobian is **additive** (chain rule for determinants):

$$\log |\det J_f(z)| = \sum_{k=1}^{K} \log |\det J_{f_k}(z_{k-1})|$$

This is the key property that makes flows scalable: as long as each individual layer has a tractable log-det, the composed flow also does.

In [ ]:
def compose_flows_forward(z, flows):
    """Apply a sequence of flows to z.

    Args:
        z     : np.ndarray (N, D)
        flows : list of (forward_fn, inverse_fn) tuples
                Each forward_fn: (z) -> (x, log_det_jac)

    Returns:
        x               : np.ndarray (N, D)
        total_log_det   : float, sum of log |det J| across all flows
    """
    # TODO:
    # x = z.copy()
    # total_log_det = 0.0
    # for fwd, _ in flows:
    #     x, log_det = fwd(x)
    #     total_log_det += log_det
    # return x, total_log_det
    x = z.copy()
    total_log_det = 0.0
    for fwd, _ in flows:
        x, log_det = fwd(x)
        total_log_det += log_det
    return x, total_log_det

In [ ]:
# Demo: compose 3 affine flows and verify additive log-det property
# Flow 1: log_scale=0.3, shift=1.0
# Flow 2: log_scale=-0.5, shift=-2.0
# Flow 3: log_scale=0.8, shift=0.5

params = [(0.3, 1.0), (-0.5, -2.0), (0.8, 0.5)]

flows_1d = [
    (
        lambda z, ls=ls, sh=sh: affine_forward_1d(z, ls, sh),
        lambda x, ls=ls, sh=sh: affine_inverse_1d(x, ls, sh)
    )
    for ls, sh in params
]

np.random.seed(1)
z_c = np.random.randn(200)
x_composed, total_ldj = compose_flows_forward(z_c, flows_1d)

# Verify: total_ldj should equal sum of individual log-dets
expected_ldj = sum(ls for ls, sh in params)
print(f'Total log-det (composed):  {total_ldj:.4f}')
print(f'Sum of individual log-dets:{expected_ldj:.4f}  (expected)')
print(f'Match: {np.isclose(total_ldj, expected_ldj)}')

# Verify: the composed mapping is affine with scale = product of scales
# x = scale1 * scale2 * scale3 * z + combined_shift
total_scale = np.prod([np.exp(ls) for ls, sh in params])
print(f'\nExpected total scale (product): {total_scale:.4f}')
print(f'exp(total_log_det):             {np.exp(total_ldj):.4f}')

---
## Part 4: 2D Affine Coupling Layer

The **affine coupling layer** (Dinh et al. 2014, NICE; Dinh et al. 2016, RealNVP) is a tractable invertible 2D transformation:

**Split** $x$ into $(x_1, x_2)$ using a binary mask:
- **Forward**: $y_1 = x_1$; $y_2 = x_2 \cdot e^{s(x_1)} + t(x_1)$
- **Inverse**: $x_1 = y_1$; $x_2 = (y_2 - t(y_1)) \cdot e^{-s(y_1)}$
- **Log** $|\det J| = \sum s(x_1)$ (only involves $s$, not $t$; triangular Jacobian)

The Jacobian is **lower-triangular** because $y_1$ only depends on $x_1$, and $y_2$ depends on $(x_1, x_2)$. A triangular matrix has determinant equal to the product of its diagonal entries.

The functions $s$ (scale) and $t$ (translation) can be **arbitrary** — even neural networks — and the layer remains invertible and tractable!

In [ ]:
def affine_coupling_forward(x, s_fn, t_fn, mask):
    """2D affine coupling layer: forward pass.

    Splits dimensions using mask (1 = pass through, 0 = transform).

    For mask=[1,0]:
        y1 = x1  (pass through)
        y2 = x2 * exp(s(x1)) + t(x1)

    Args:
        x    : np.ndarray (N, 2)
        s_fn : callable, (N, 1) -> (N, 1), scale function (applied to masked input)
        t_fn : callable, (N, 1) -> (N, 1), translation function
        mask : np.ndarray (2,), binary mask e.g. [1, 0] or [0, 1]

    Returns:
        y           : np.ndarray (N, 2)
        log_det_jac : np.ndarray (N,), per-sample log |det J|
    """
    # TODO:
    # For mask=[1,0]: x1 (dim 0) passes through; x2 (dim 1) is transformed
    # pass_idx  = np.where(mask == 1)[0][0]   # index of pass-through dim
    # trans_idx = np.where(mask == 0)[0][0]   # index of transformed dim
    # s_val = s_fn(x[:, [pass_idx]])           # shape (N, 1)
    # t_val = t_fn(x[:, [pass_idx]])           # shape (N, 1)
    # y = x.copy()
    # y[:, trans_idx] = x[:, trans_idx] * np.exp(s_val[:, 0]) + t_val[:, 0]
    # log_det_jac = s_val[:, 0]               # shape (N,)
    # return y, log_det_jac
    pass_idx  = np.where(mask == 1)[0][0]
    trans_idx = np.where(mask == 0)[0][0]
    s_val = s_fn(x[:, [pass_idx]])
    t_val = t_fn(x[:, [pass_idx]])
    y = x.copy()
    y[:, trans_idx] = x[:, trans_idx] * np.exp(s_val[:, 0]) + t_val[:, 0]
    log_det_jac = s_val[:, 0]
    return y, log_det_jac


def affine_coupling_inverse(y, s_fn, t_fn, mask):
    """2D affine coupling layer: inverse pass.

    For mask=[1,0]:
        x1 = y1  (pass through)
        x2 = (y2 - t(y1)) * exp(-s(y1))

    Args:
        y    : np.ndarray (N, 2)
        s_fn : callable, (N, 1) -> (N, 1)
        t_fn : callable, (N, 1) -> (N, 1)
        mask : np.ndarray (2,)

    Returns:
        x           : np.ndarray (N, 2)
        log_det_jac : np.ndarray (N,), log |det J_{inv}| = -log_det_fwd
    """
    # TODO:
    # pass_idx  = np.where(mask == 1)[0][0]
    # trans_idx = np.where(mask == 0)[0][0]
    # s_val = s_fn(y[:, [pass_idx]])
    # t_val = t_fn(y[:, [pass_idx]])
    # x = y.copy()
    # x[:, trans_idx] = (y[:, trans_idx] - t_val[:, 0]) * np.exp(-s_val[:, 0])
    # return x, -s_val[:, 0]
    pass_idx  = np.where(mask == 1)[0][0]
    trans_idx = np.where(mask == 0)[0][0]
    s_val = s_fn(y[:, [pass_idx]])
    t_val = t_fn(y[:, [pass_idx]])
    x = y.copy()
    x[:, trans_idx] = (y[:, trans_idx] - t_val[:, 0]) * np.exp(-s_val[:, 0])
    return x, -s_val[:, 0]

In [ ]:
# Sanity check: forward then inverse should recover original input
# Use simple linear s_fn and t_fn for the numpy demo
np.random.seed(3)
x_2d = np.random.randn(50, 2)
mask_10 = np.array([1, 0])

# Linear s_fn: s(x1) = 0.5 * x1, t_fn: t(x1) = -0.3 * x1
s_fn_linear = lambda u: 0.5 * u
t_fn_linear = lambda u: -0.3 * u

y_2d, ldj_fwd_2d = affine_coupling_forward(x_2d, s_fn_linear, t_fn_linear, mask_10)
x_rec_2d, ldj_inv_2d = affine_coupling_inverse(y_2d, s_fn_linear, t_fn_linear, mask_10)

print(f'Roundtrip max error: {np.max(np.abs(x_2d - x_rec_2d)):.2e}  (should be ~0)')
print(f'log_det_fwd (first 5): {ldj_fwd_2d[:5].round(4)}')
print(f'log_det_inv (first 5): {ldj_inv_2d[:5].round(4)}  (should be negatives of above)')
print(f'Sum fwd+inv (first 5): {(ldj_fwd_2d + ldj_inv_2d)[:5].round(6)}')

# Verify: x1 (dim 0) is unchanged (pass-through)
print(f'\nDim 0 unchanged: {np.allclose(x_2d[:, 0], y_2d[:, 0])}  (mask=[1,0] -> dim 0 passes through)')

In [ ]:
# Visualise the effect of the coupling layer on a 2D grid
g = np.linspace(-3, 3, 20)
G1, G2 = np.meshgrid(g, g)
x_grid_2d = np.stack([G1.ravel(), G2.ravel()], axis=1)

y_grid_2d, _ = affine_coupling_forward(x_grid_2d, s_fn_linear, t_fn_linear, mask_10)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(x_grid_2d[:, 0], x_grid_2d[:, 1], s=15, alpha=0.6, color='tab:blue')
axes[0].set_title('Input grid (z)')
axes[0].set_xlabel('z1'); axes[0].set_ylabel('z2')
axes[0].set_aspect('equal'); axes[0].grid(True, alpha=0.3)

axes[1].scatter(y_grid_2d[:, 0], y_grid_2d[:, 1], s=15, alpha=0.6, color='tab:orange')
axes[1].set_title('Output grid after coupling layer (x)')
axes[1].set_xlabel('x1'); axes[1].set_ylabel('x2')
axes[1].set_aspect('equal'); axes[1].grid(True, alpha=0.3)

plt.suptitle('2D affine coupling layer: mask=[1,0], s(x1)=0.5x1, t(x1)=-0.3x1')
plt.tight_layout()
plt.show()

---
## Part 5: RealNVP — Stacking Coupling Layers (PyTorch)

**RealNVP** (Dinh et al. 2016) builds a powerful normalising flow by stacking affine coupling layers with **alternating masks**:
- Layer 1: mask $= [1, 0]$ → transforms $x_2$ conditioned on $x_1$
- Layer 2: mask $= [0, 1]$ → transforms $x_1$ conditioned on $x_2$
- Repeat ...

Alternating masks ensure that **all dimensions** get transformed through the stack, despite each layer leaving one dimension unchanged.

The scale ($s$) and translation ($t$) functions are small **MLPs** shared within each coupling layer. The network can be arbitrarily expressive while the flow remains invertible and its log-det tractable.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

device = 'mps' if torch.backends.mps.is_available() else 'cpu'
print(f'Using device: {device}')

In [ ]:
class CouplingLayer(nn.Module):
    """Affine coupling layer for 2D data.

    mask=[1,0]: x1 -> (s,t) -> transform x2
    mask=[0,1]: x2 -> (s,t) -> transform x1

    s and t are small MLPs: Linear(1, H) -> ReLU -> Linear(H, H) -> ReLU -> Linear(H, 1)
    """

    def __init__(self, mask, hidden_dim=64):
        """
        Args:
            mask       : list of int, e.g. [1, 0] or [0, 1]
            hidden_dim : int, MLP hidden width
        """
        super().__init__()
        self.register_buffer('mask', torch.tensor(mask, dtype=torch.float32))
        # TODO:
        # self.s_net = nn.Sequential(
        #     nn.Linear(1, hidden_dim), nn.ReLU(),
        #     nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
        #     nn.Linear(hidden_dim, 1)
        # )
        # self.t_net = nn.Sequential(
        #     nn.Linear(1, hidden_dim), nn.ReLU(),
        #     nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
        #     nn.Linear(hidden_dim, 1)
        # )
        raise NotImplementedError

    def forward(self, x):
        """Forward pass: z -> x direction.

        Args:
            x : Tensor (B, 2)

        Returns:
            y           : Tensor (B, 2)
            log_det_jac : Tensor (B,)
        """
        # TODO: implement affine coupling forward in PyTorch
        # Hint:
        # pass_idx  = int(self.mask[0].item())  # 0 or 1
        # trans_idx = 1 - pass_idx
        # s_val = self.s_net(x[:, [pass_idx]])   # (B, 1)
        # t_val = self.t_net(x[:, [pass_idx]])   # (B, 1)
        # y = x.clone()
        # y[:, trans_idx] = x[:, trans_idx] * torch.exp(s_val[:, 0]) + t_val[:, 0]
        # log_det_jac = s_val[:, 0]              # (B,)
        # return y, log_det_jac
        raise NotImplementedError

    def inverse(self, y):
        """Inverse pass: x -> z direction.

        Args:
            y : Tensor (B, 2)

        Returns:
            x           : Tensor (B, 2)
            log_det_jac : Tensor (B,),  negative of forward log-det
        """
        # TODO: implement affine coupling inverse
        # Hint:
        # pass_idx  = int(self.mask[0].item())
        # trans_idx = 1 - pass_idx
        # s_val = self.s_net(y[:, [pass_idx]])
        # t_val = self.t_net(y[:, [pass_idx]])
        # x = y.clone()
        # x[:, trans_idx] = (y[:, trans_idx] - t_val[:, 0]) * torch.exp(-s_val[:, 0])
        # return x, -s_val[:, 0]
        raise NotImplementedError

In [ ]:
class RealNVP(nn.Module):
    """RealNVP: stack of alternating coupling layers.

    Flow direction: z -> [CouplingLayer1] -> [CouplingLayer2] -> ... -> x
    Alternating masks: [1,0], [0,1], [1,0], [0,1], ...
    """

    def __init__(self, n_layers=4, hidden_dim=64):
        """
        Args:
            n_layers   : int, number of coupling layers (should be even)
            hidden_dim : int, MLP hidden width in each coupling layer
        """
        super().__init__()
        masks = [[1, 0], [0, 1]] * (n_layers // 2)
        self.layers = nn.ModuleList([
            CouplingLayer(mask, hidden_dim) for mask in masks
        ])

    def forward(self, z):
        """Map from base distribution to data space.

        Args:
            z : Tensor (B, 2)

        Returns:
            x           : Tensor (B, 2)
            log_det_jac : Tensor (B,), total log |det J|
        """
        # TODO:
        # x = z
        # total_log_det = torch.zeros(z.shape[0], device=z.device)
        # for layer in self.layers:
        #     x, log_det = layer.forward(x)
        #     total_log_det = total_log_det + log_det
        # return x, total_log_det
        raise NotImplementedError

    def inverse(self, x):
        """Map from data space to base distribution.

        Args:
            x : Tensor (B, 2)

        Returns:
            z           : Tensor (B, 2)
            log_det_jac : Tensor (B,), total log |det J_{f^{-1}}|
        """
        # TODO: apply layers in reverse order using layer.inverse()
        # z = x
        # total_log_det = torch.zeros(x.shape[0], device=x.device)
        # for layer in reversed(self.layers):
        #     z, log_det = layer.inverse(z)
        #     total_log_det = total_log_det + log_det
        # return z, total_log_det
        raise NotImplementedError

    def log_prob(self, x):
        """Compute log p_X(x) = log p_Z(z) + log |det J_{f^{-1}}(x)|.

        Args:
            x : Tensor (B, 2)

        Returns:
            log_prob : Tensor (B,)
        """
        # TODO:
        # z, log_det_inv = self.inverse(x)
        # D = x.shape[1]  # = 2
        # log_pz = -0.5 * (z ** 2).sum(dim=1) - D / 2 * torch.log(torch.tensor(2 * torch.pi))
        # return log_pz + log_det_inv
        raise NotImplementedError

In [ ]:
# Sanity check: model instantiation and forward/inverse shapes
# (Requires CouplingLayer to be implemented — will raise NotImplementedError otherwise)
torch.manual_seed(0)
try:
    _model_test = RealNVP(n_layers=4, hidden_dim=32).to(device)
    _z_test = torch.randn(8, 2).to(device)

    _x_test, _ldj_fwd = _model_test.forward(_z_test)
    print(f'forward: z shape={_z_test.shape} -> x shape={_x_test.shape}, ldj shape={_ldj_fwd.shape}')

    _z_rec, _ldj_inv = _model_test.inverse(_x_test)
    print(f'inverse: x shape={_x_test.shape} -> z shape={_z_rec.shape}')

    roundtrip_err = (_z_test - _z_rec).abs().max().item()
    print(f'Roundtrip max error: {roundtrip_err:.2e}  (should be ~0)')

    _lp = _model_test.log_prob(_x_test)
    print(f'log_prob shape: {_lp.shape}  (expected (8,))')
    print('All checks passed!')
except NotImplementedError:
    print('NotImplementedError: implement CouplingLayer and RealNVP methods first.')

---
## Part 6: Training by NLL

Normalising flows are trained by **maximum likelihood estimation** — equivalently, minimising negative log-likelihood (NLL):

$$\mathcal{L} = -\frac{1}{N} \sum_{i=1}^N \log p_X(x^{(i)})$$

Each forward pass through `model.log_prob(x)` gives exact log-likelihoods (no ELBO approximation needed). Gradients flow through the log-det-Jacobian terms into the MLP parameters.

We train on the **two moons** dataset — a standard 2D benchmark for density estimation.

In [ ]:
from sklearn.datasets import make_moons

np.random.seed(42)
X_moons, _ = make_moons(n_samples=2000, noise=0.05)

# Standardise to roughly N(0, 1)
X_moons = (X_moons - X_moons.mean(axis=0)) / X_moons.std(axis=0)
X_moons = X_moons.astype(np.float32)

plt.figure()
plt.scatter(X_moons[:, 0], X_moons[:, 1], s=8, alpha=0.5, color='tab:blue')
plt.title('Training data: two moons')
plt.xlabel('x1'); plt.ylabel('x2')
plt.axis('equal')
plt.grid(True, alpha=0.3)
plt.show()

print(f'Dataset shape: {X_moons.shape}')
print(f'Mean: {X_moons.mean(axis=0).round(3)},  Std: {X_moons.std(axis=0).round(3)}')

In [ ]:
def train_realnvp(X_data, n_layers=4, n_epochs=300, lr=1e-3, batch_size=256, hidden_dim=64):
    """Train RealNVP on 2D data by minimising NLL.

    Loss = -mean log_prob(x)

    Args:
        X_data     : np.ndarray (N, 2)
        n_layers   : int, number of coupling layers
        n_epochs   : int
        lr         : float, Adam learning rate
        batch_size : int
        hidden_dim : int, MLP hidden width

    Returns:
        model        : trained RealNVP
        loss_history : list of float, per-epoch average NLL
    """
    X_t = torch.tensor(X_data, dtype=torch.float32).to(device)
    model = RealNVP(n_layers=n_layers, hidden_dim=hidden_dim).to(device)
    optimiser = torch.optim.Adam(model.parameters(), lr=lr)

    dataset = torch.utils.data.TensorDataset(X_t)
    loader  = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)

    loss_history = []
    for epoch in range(n_epochs):
        epoch_loss = 0.0
        for (x_batch,) in loader:
            # TODO:
            # 1. optimiser.zero_grad()
            # 2. loss = -model.log_prob(x_batch).mean()
            # 3. loss.backward()
            # 4. optimiser.step()
            # 5. epoch_loss += loss.item()
            raise NotImplementedError
        loss_history.append(epoch_loss / len(loader))
        if (epoch + 1) % 50 == 0:
            print(f'Epoch {epoch+1}/{n_epochs}  NLL={loss_history[-1]:.3f}')
    return model, loss_history

In [ ]:
# Train RealNVP — complete CouplingLayer, RealNVP, and train_realnvp TODO stubs first!
torch.manual_seed(42)
try:
    model_nvp, loss_history_nvp = train_realnvp(
        X_moons, n_layers=4, n_epochs=300, lr=1e-3, batch_size=256, hidden_dim=64
    )

    plt.figure()
    plt.plot(loss_history_nvp, color='tab:blue')
    plt.xlabel('Epoch')
    plt.ylabel('NLL')
    plt.title('RealNVP training: NLL loss')
    plt.grid(True, alpha=0.3)
    plt.show()
except NotImplementedError:
    print('NotImplementedError: complete all TODO stubs before training.')

---
## Part 7: Sampling and Density Visualisation

Generating samples from a normalising flow is simple:
1. Sample $z \sim \mathcal{N}(0, I)$ from the base distribution
2. Pass through the **forward** mapping $x = f(z)$

For density evaluation on a grid, use `model.log_prob(x)` — this is exact, no approximation needed.

In [ ]:
def sample_realnvp(model, n_samples):
    """Generate samples from the trained RealNVP.

    Steps:
        1. Sample z ~ N(0, I)  — base distribution
        2. Pass through model.forward(z) to get x

    Args:
        model     : trained RealNVP
        n_samples : int

    Returns:
        samples : np.ndarray (n_samples, 2)
    """
    model.eval()
    with torch.no_grad():
        # TODO:
        # z = torch.randn(n_samples, 2).to(device)
        # x, _ = model.forward(z)
        # return x.cpu().numpy()
        raise NotImplementedError

In [ ]:
# Generate samples and compare to training data
try:
    np.random.seed(0)
    samples_nvp = sample_realnvp(model_nvp, n_samples=2000)

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    axes[0].scatter(X_moons[:, 0], X_moons[:, 1], s=8, alpha=0.4, color='tab:blue')
    axes[0].set_title('Training data (two moons)')
    axes[0].set_xlabel('x1'); axes[0].set_ylabel('x2')
    axes[0].set_aspect('equal'); axes[0].grid(True, alpha=0.3)

    axes[1].scatter(samples_nvp[:, 0], samples_nvp[:, 1], s=8, alpha=0.5, color='tab:orange')
    axes[1].set_title('RealNVP generated samples')
    axes[1].set_xlabel('x1'); axes[1].set_ylabel('x2')
    axes[1].set_aspect('equal'); axes[1].grid(True, alpha=0.3)

    plt.suptitle('Training data vs. RealNVP samples')
    plt.tight_layout()
    plt.show()
except NotImplementedError:
    print('NotImplementedError: complete sample_realnvp first.')
except NameError:
    print('NameError: train the model first (run the training cell).')

In [ ]:
# Plot learned log-density as a heatmap over a 2D grid
try:
    model_nvp.eval()
    g_vals = np.linspace(-3, 3, 100)
    G1, G2 = np.meshgrid(g_vals, g_vals)
    X_grid = np.stack([G1.ravel(), G2.ravel()], axis=1).astype(np.float32)

    X_grid_t = torch.tensor(X_grid).to(device)
    with torch.no_grad():
        log_prob_grid = model_nvp.log_prob(X_grid_t).cpu().numpy()

    log_prob_map = log_prob_grid.reshape(100, 100)

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    im = axes[0].contourf(G1, G2, np.exp(log_prob_map), levels=50, cmap='viridis')
    plt.colorbar(im, ax=axes[0])
    axes[0].set_title('Learned density $p_X(x)$')
    axes[0].set_xlabel('x1'); axes[0].set_ylabel('x2')
    axes[0].set_aspect('equal')

    im2 = axes[1].contourf(G1, G2, log_prob_map, levels=50, cmap='plasma')
    plt.colorbar(im2, ax=axes[1])
    axes[1].scatter(X_moons[:500, 0], X_moons[:500, 1], s=5, alpha=0.3, color='white', label='Data')
    axes[1].set_title('Log-density $\\log p_X(x)$ with data overlay')
    axes[1].set_xlabel('x1'); axes[1].set_ylabel('x2')
    axes[1].set_aspect('equal')
    axes[1].legend()

    plt.suptitle('RealNVP learned density')
    plt.tight_layout()
    plt.show()
except NotImplementedError:
    print('NotImplementedError: complete the RealNVP implementation first.')
except NameError:
    print('NameError: train the model first.')

**Reflection question 1**: Why must all layers of a normalising flow be invertible? What goes wrong if they aren't?

**Your answer**: TODO

**Reflection question 2**: The Jacobian of an affine coupling layer is triangular. Why does this make computing $\log |\det J|$ cheap?

**Your answer**: TODO

---
## Summary

In this notebook you:

- Derived and implemented the **change of variables formula** for exact density computation
- Built **1D affine flows** with forward, inverse, and log-likelihood, and fit them to data by grid search
- Composed multiple flows and verified the **additive log-det property** (chain rule for Jacobians)
- Implemented **2D affine coupling layers** (forward + inverse) exploiting the triangular Jacobian structure
- Built and trained a **RealNVP model** on 2D two-moons data using exact NLL minimisation
- Generated new samples and visualised the **learned density** as a heatmap

| Component | Role |
|---|---|
| `change_of_variables_log_prob` | Core formula: log p_X = log p_Z - log\|det J_f\| |
| `affine_forward_1d / inverse_1d` | Simplest invertible flow with tractable log-det |
| `flow_log_prob_1d` | Exact likelihood under a 1D affine flow |
| `compose_flows_forward` | Stack flows; log-dets add up |
| `affine_coupling_forward / inverse` | 2D coupling: triangular Jacobian, arbitrary s and t |
| `CouplingLayer` | PyTorch coupling layer with MLP scale and shift |
| `RealNVP` | Stack of alternating coupling layers |
| `train_realnvp` | MLE training: minimise exact NLL |
| `sample_realnvp` | Sample: z ~ N(0,I), push through forward flow |

**Next**: Assignment 9 extends flows to continuous time using neural ODEs and introduces the flow matching training paradigm.